# 04 - Feature Engineering

## Analytical Objectives

- Create calendar-based features from the Date column.
- Generate lag features using historical sales.
- Create rolling statistical features.
- Create expanding statistical features.
- Generate holiday-related features.
- Create store-level and department-level aggregate features.
- Apply cyclical encoding to seasonal variables.
- Handle missing values introduced during feature engineering.
- Select the final features for model training.
- Save the feature-engineered dataset.

## Expected Outcome

By the end of this notebook, we will have:

- A feature-rich dataset optimized for time series forecasting.
- Historical sales patterns represented through lag and rolling features.
- Temporal and seasonal information encoded for machine learning models.
- A final modeling dataset ready for baseline and advanced forecasting models.

#### Load Libraries

In [81]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

#### Load Dataset

In [82]:
train = pd.read_csv("../data/processed/train_clean.csv")

train["Date"] = pd.to_datetime(train["Date"])

train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106


In [83]:
test = pd.read_csv(
    "../data/raw/test.csv"
)

test["Date"] = pd.to_datetime(test["Date"])
test.head()

,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


In [84]:
features = pd.read_csv(
    "../data/raw/features.csv"
)

features["Date"] = pd.to_datetime(test["Date"])
features.head()

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2012-11-02,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2012-11-09,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2012-11-16,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2012-11-23,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2012-11-30,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [85]:
stores = pd.read_csv(
    "../data/raw/stores.csv"
)

stores["Date"] = pd.to_datetime(test["Date"])
stores.head()

,Store,Type,Size,Date
0,1,A,151315,2012-11-02
1,2,A,202307,2012-11-09
2,3,B,37392,2012-11-16
3,4,A,205863,2012-11-23
4,5,B,34875,2012-11-30


#### Date Features

In [86]:
train["Year"] = train["Date"].dt.year

train["Quarter"] = train["Date"].dt.quarter

train["Month"] = train["Date"].dt.month

train["Week"] = train["Date"].dt.isocalendar().week.astype(int)

train["Day"] = train["Date"].dt.day

train["DayOfWeek"] = train["Date"].dt.dayofweek

train["IsWeekend"] = (
    train["DayOfWeek"] >= 5
).astype(int)

In [87]:
#Verify New Features

train[
    [
        "Date",
        "Year",
        "Quarter",
        "Month",
        "Week",
        "Day",
        "DayOfWeek",
        "IsWeekend"
    ]
].head()

,Date,Year,Quarter,Month,Week,Day,DayOfWeek,IsWeekend
0,2010-02-05,2010,1,2,5,5,4,0
1,2010-02-12,2010,1,2,6,12,4,0
2,2010-02-19,2010,1,2,7,19,4,0
3,2010-02-26,2010,1,2,8,26,4,0
4,2010-03-05,2010,1,3,9,5,4,0


#### Lag Features

In [88]:
#Dataset Sort

train = train.sort_values(
    by=["Store", "Dept", "Date"]
)

In [89]:
#1-Week Lag

train["Lag_1"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .shift(1)
)

In [90]:
#2-Week Lag

train["Lag_2"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .shift(2)
)

In [91]:
#4-Week Lag

train["Lag_4"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .shift(4)
)

In [92]:
#8-Week Lag

train["Lag_8"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .shift(8)
)

In [93]:
#12-Week Lag

train["Lag_12"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .shift(12)
)

In [94]:
#52-Week Lag

train["Lag_52"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .shift(52)
)

In [95]:
#Verify Lag Features

train[
    [
        "Store",
        "Dept",
        "Date",
        "Weekly_Sales",
        "Lag_1",
        "Lag_2",
        "Lag_4",
        "Lag_8",
        "Lag_12",
        "Lag_52"
    ]
].head(15)

,Store,Dept,Date,Weekly_Sales,Lag_1,Lag_2,Lag_4,Lag_8,Lag_12,Lag_52
0,1,1,2010-02-05,24924.50,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,2010-02-12,46039.49,24924.50,NaN,NaN,NaN,NaN,NaN
2,1,1,2010-02-19,41595.55,46039.49,24924.50,NaN,NaN,NaN,NaN
3,1,1,2010-02-26,19403.54,41595.55,46039.49,NaN,NaN,NaN,NaN
4,1,1,2010-03-05,21827.90,19403.54,41595.55,24924.50,NaN,NaN,NaN
5,1,1,2010-03-12,21043.39,21827.90,19403.54,46039.49,NaN,NaN,NaN
6,1,1,2010-03-19,22136.64,21043.39,21827.90,41595.55,NaN,NaN,NaN
7,1,1,2010-03-26,26229.21,22136.64,21043.39,19403.54,NaN,NaN,NaN
8,1,1,2010-04-02,57258.43,26229.21,22136.64,21827.90,24924.50,NaN,NaN
9,1,1,2010-04-09,42960.91,57258.43,26229.21,21043.39,46039.49,NaN,NaN


In [96]:
#Missing Values Introduced by Lag Features

lag_columns = [
    "Lag_1",
    "Lag_2",
    "Lag_4",
    "Lag_8",
    "Lag_12",
    "Lag_52"
]

train[lag_columns].isnull().sum()

Lag_1       3331
Lag_2       6625
Lag_4      13134
Lag_8      25966
Lag_12     38615
Lag_52    160487
dtype: int64

In [97]:
#Preview Lag Features

train.loc[
    train["Store"].eq(1) & train["Dept"].eq(1),
    [
        "Date",
        "Weekly_Sales",
        "Lag_1",
        "Lag_2",
        "Lag_4",
        "Lag_8",
        "Lag_12",
        "Lag_52"
    ]
].head(15)

,Date,Weekly_Sales,Lag_1,Lag_2,Lag_4,Lag_8,Lag_12,Lag_52
0,2010-02-05,24924.50,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-12,46039.49,24924.50,NaN,NaN,NaN,NaN,NaN
2,2010-02-19,41595.55,46039.49,24924.50,NaN,NaN,NaN,NaN
3,2010-02-26,19403.54,41595.55,46039.49,NaN,NaN,NaN,NaN
4,2010-03-05,21827.90,19403.54,41595.55,24924.50,NaN,NaN,NaN
5,2010-03-12,21043.39,21827.90,19403.54,46039.49,NaN,NaN,NaN
6,2010-03-19,22136.64,21043.39,21827.90,41595.55,NaN,NaN,NaN
7,2010-03-26,26229.21,22136.64,21043.39,19403.54,NaN,NaN,NaN
8,2010-04-02,57258.43,26229.21,22136.64,21827.90,24924.50,NaN,NaN
9,2010-04-09,42960.91,57258.43,26229.21,21043.39,46039.49,NaN,NaN


#### Rolling Features

In [98]:
#4-Week Rolling Mean

train["Rolling_Mean_4"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .transform(lambda x: x.shift(1).rolling(window=4).mean())
)

In [99]:
#8-Week Rolling Mean

train["Rolling_Mean_8"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .transform(lambda x: x.shift(1).rolling(window=8).mean())
)

In [100]:
#12-Week Rolling Mean

train["Rolling_Mean_12"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .transform(lambda x: x.shift(1).rolling(window=12).mean())
)

In [101]:
#4-Week Rolling Standard Deviation

train["Rolling_Std_4"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .transform(lambda x: x.shift(1).rolling(window=4).std())
)

In [102]:
#8-Week Rolling Standard Deviation

train["Rolling_Std_8"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .transform(lambda x: x.shift(1).rolling(window=8).std())
)

In [103]:
#12-Week Rolling Standard Deviation

train["Rolling_Std_12"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .transform(lambda x: x.shift(1).rolling(window=12).std())
)

In [104]:
#4-Week Rolling Minimum

train["Rolling_Min_4"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .transform(lambda x: x.shift(1).rolling(window=4).min())
)

In [105]:
#4-Week Rolling Maximum

train["Rolling_Max_4"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .transform(lambda x: x.shift(1).rolling(window=4).max())
)

In [106]:
#Verify Rolling Features

train.loc[
    (train["Store"] == 1) & (train["Dept"] == 1),
    [
        "Date",
        "Weekly_Sales",
        "Rolling_Mean_4",
        "Rolling_Mean_8",
        "Rolling_Mean_12",
        "Rolling_Std_4",
        "Rolling_Std_8",
        "Rolling_Std_12",
        "Rolling_Min_4",
        "Rolling_Max_4"
    ]
].head(20)

,Date,Weekly_Sales,Rolling_Mean_4,Rolling_Mean_8,Rolling_Mean_12,Rolling_Std_4,Rolling_Std_8,Rolling_Std_12,Rolling_Min_4,Rolling_Max_4
0,2010-02-05,24924.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-02-12,46039.49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-02-19,41595.55,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-02-26,19403.54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-03-05,21827.90,32990.7700,NaN,NaN,12832.106391,NaN,NaN,19403.54,46039.49
5,2010-03-12,21043.39,32216.6200,NaN,NaN,13554.047185,NaN,NaN,19403.54,46039.49
6,2010-03-19,22136.64,25967.5950,NaN,NaN,10467.484020,NaN,NaN,19403.54,41595.55
7,2010-03-26,26229.21,21102.8675,NaN,NaN,1222.784968,NaN,NaN,19403.54,22136.64
8,2010-04-02,57258.43,22809.2850,27900.02750,NaN,2325.929203,10124.538627,NaN,21043.39,26229.21
9,2010-04-09,42960.91,31666.9175,31941.76875,NaN,17206.391261,14342.348043,NaN,21043.39,57258.43


In [107]:
#Missing Values

rolling_columns = [
    "Rolling_Mean_4",
    "Rolling_Mean_8",
    "Rolling_Mean_12",
    "Rolling_Std_4",
    "Rolling_Std_8",
    "Rolling_Std_12",
    "Rolling_Min_4",
    "Rolling_Max_4"
]

train[rolling_columns].isnull().sum()

Rolling_Mean_4     13134
Rolling_Mean_8     25966
Rolling_Mean_12    38615
Rolling_Std_4      13134
Rolling_Std_8      25966
Rolling_Std_12     38615
Rolling_Min_4      13134
Rolling_Max_4      13134
dtype: int64

#### Creating Expanding Features

In [108]:
#Expanding Mean

train["Expanding_Mean"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .transform(lambda x: x.shift(1).expanding().mean())
)

In [109]:
#Expanding Standard Deviation

train["Expanding_Std"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .transform(lambda x: x.shift(1).expanding().std())
)

In [110]:
#Expanding Minimum

train["Expanding_Min"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .transform(lambda x: x.shift(1).expanding().min())
)

In [111]:
#Expanding Maximum

train["Expanding_Max"] = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"]
         .transform(lambda x: x.shift(1).expanding().max())
)

In [112]:
#Verify Expanding Features

train.loc[
    (train["Store"] == 1) &
    (train["Dept"] == 1),
    [
        "Date",
        "Weekly_Sales",
        "Expanding_Mean",
        "Expanding_Std",
        "Expanding_Min",
        "Expanding_Max"
    ]
].head(20)

,Date,Weekly_Sales,Expanding_Mean,Expanding_Std,Expanding_Min,Expanding_Max
0,2010-02-05,24924.50,NaN,NaN,NaN,NaN
1,2010-02-12,46039.49,24924.500000,NaN,24924.50,24924.50
2,2010-02-19,41595.55,35481.995000,14930.552614,24924.50,46039.49
3,2010-02-26,19403.54,37519.846667,11131.900957,24924.50,46039.49
4,2010-03-05,21827.90,32990.770000,12832.106391,19403.54,46039.49
5,2010-03-12,21043.39,30758.196000,12182.739805,19403.54,46039.49
6,2010-03-19,22136.64,29139.061667,11595.899933,19403.54,46039.49
7,2010-03-26,26229.21,28138.715714,10911.412537,19403.54,46039.49
8,2010-04-02,57258.43,27900.027500,10124.538627,19403.54,46039.49
9,2010-04-09,42960.91,31162.072222,13618.422046,19403.54,57258.43


In [113]:
#Check Missing Values

expanding_columns = [
    "Expanding_Mean",
    "Expanding_Std",
    "Expanding_Min",
    "Expanding_Max"
]

train[expanding_columns].isnull().sum()

Expanding_Mean    3331
Expanding_Std     6625
Expanding_Min     3331
Expanding_Max     3331
dtype: int64

In [114]:
#Compare Rolling vs Expanding Features

train.loc[
    (train["Store"] == 1) &
    (train["Dept"] == 1),
    [
        "Date",
        "Weekly_Sales",
        "Rolling_Mean_4",
        "Rolling_Mean_12",
        "Expanding_Mean"
    ]
].head(20)

,Date,Weekly_Sales,Rolling_Mean_4,Rolling_Mean_12,Expanding_Mean
0,2010-02-05,24924.50,NaN,NaN,NaN
1,2010-02-12,46039.49,NaN,NaN,24924.500000
2,2010-02-19,41595.55,NaN,NaN,35481.995000
3,2010-02-26,19403.54,NaN,NaN,37519.846667
4,2010-03-05,21827.90,32990.7700,NaN,32990.770000
5,2010-03-12,21043.39,32216.6200,NaN,30758.196000
6,2010-03-19,22136.64,25967.5950,NaN,29139.061667
7,2010-03-26,26229.21,21102.8675,NaN,28138.715714
8,2010-04-02,57258.43,22809.2850,NaN,27900.027500
9,2010-04-09,42960.91,31666.9175,NaN,31162.072222


#### Holiday Features

In [115]:
#Holiday Indicator

train["Holiday_Flag"] = train["IsHoliday"].astype(int)

In [116]:
#Previous Week Was a Holiday

train["Previous_Holiday"] = (
    train.groupby(["Store", "Dept"])["Holiday_Flag"]
         .shift(1)
         .fillna(0)
         .astype(int)
)

In [117]:
#Next Week Is a Holiday

train["Next_Holiday"] = (
    train.groupby(["Store", "Dept"])["Holiday_Flag"]
         .shift(-1)
         .fillna(0)
         .astype(int)
)

In [118]:
#Consecutive Holiday Indicator

train["Near_Holiday"] = (
    (
        (train["Holiday_Flag"] == 1) |
        (train["Previous_Holiday"] == 1) |
        (train["Next_Holiday"] == 1)
    )
).astype(int)

In [119]:
#Verify Holiday Features

train.loc[
    train["Holiday_Flag"] == 1,
    [
        "Date",
        "Store",
        "Dept",
        "Holiday_Flag",
        "Previous_Holiday",
        "Next_Holiday",
        "Near_Holiday"
    ]
].head(20)

,Date,Store,Dept,Holiday_Flag,Previous_Holiday,Next_Holiday,Near_Holiday
1,2010-02-12,1,1,1,0,0,1
31,2010-09-10,1,1,1,0,0,1
42,2010-11-26,1,1,1,0,0,1
47,2010-12-31,1,1,1,0,0,1
53,2011-02-11,1,1,1,0,0,1
83,2011-09-09,1,1,1,0,0,1
94,2011-11-25,1,1,1,0,0,1
99,2011-12-30,1,1,1,0,0,1
105,2012-02-10,1,1,1,0,0,1
135,2012-09-07,1,1,1,0,0,1


In [120]:
#Distribution of Holiday Features

holiday_features = [
    "Holiday_Flag",
    "Previous_Holiday",
    "Next_Holiday",
    "Near_Holiday"
]

train[holiday_features].sum()

Holiday_Flag        29661
Previous_Holiday    29635
Next_Holiday        29575
Near_Holiday        88780
dtype: int64

In [121]:
#Check Missing Values

train[
    [
        "Holiday_Flag",
        "Previous_Holiday",
        "Next_Holiday",
        "Near_Holiday"
    ]
].isnull().sum()

Holiday_Flag        0
Previous_Holiday    0
Next_Holiday        0
Near_Holiday        0
dtype: int64

#### Store Features

In [122]:
#Average Sales by Store

store_stats = (
    train.groupby("Store")
         .agg(
             Store_Avg_Sales=("Weekly_Sales", "mean"),
             Store_Std_Sales=("Weekly_Sales", "std"),
             Store_Min_Sales=("Weekly_Sales", "min"),
             Store_Max_Sales=("Weekly_Sales", "max")
         )
         .reset_index()
)

store_stats.head()

,Store,Store_Avg_Sales,Store_Std_Sales,Store_Min_Sales,Store_Max_Sales
0,1,21710.543621,27748.945511,-863.00,203670.47
1,2,26898.070031,33077.612059,-1098.00,285353.53
2,3,6373.033983,14251.034807,-1008.96,155897.94
3,4,29161.210415,34583.677814,-898.00,385051.04
4,5,5053.415813,8068.221050,-101.26,93517.72


In [123]:
#Merge Store Statistics

train = train.merge(
    store_stats,
    on="Store",
    how="left"
)

train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Year,Quarter,Month,Week,Day,DayOfWeek,IsWeekend,Lag_1,Lag_2,Lag_4,Lag_8,Lag_12,Lag_52,Rolling_Mean_4,Rolling_Mean_8,Rolling_Mean_12,Rolling_Std_4,Rolling_Std_8,Rolling_Std_12,Rolling_Min_4,Rolling_Max_4,Expanding_Mean,Expanding_Std,Expanding_Min,Expanding_Max,Holiday_Flag,Previous_Holiday,Next_Holiday,Near_Holiday,Store_Avg_Sales,Store_Std_Sales,Store_Min_Sales,Store_Max_Sales
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,2010,1,2,5,5,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,1,21710.543621,27748.945511,-863.0,203670.47
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,2010,1,2,6,12,4,0,24924.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24924.500000,NaN,24924.50,24924.50,1,0,0,1,21710.543621,27748.945511,-863.0,203670.47
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,2010,1,2,7,19,4,0,46039.49,24924.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35481.995000,14930.552614,24924.50,46039.49,0,1,0,1,21710.543621,27748.945511,-863.0,203670.47
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,2010,1,2,8,26,4,0,41595.55,46039.49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37519.846667,11131.900957,24924.50,46039.49,0,0,0,0,21710.543621,27748.945511,-863.0,203670.47
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,2010,1,3,9,5,4,0,19403.54,41595.55,24924.5,NaN,NaN,NaN,32990.77,NaN,NaN,12832.106391,NaN,NaN,19403.54,46039.49,32990.770000,12832.106391,19403.54,46039.49,0,0,0,0,21710.543621,27748.945511,-863.0,203670.47


In [124]:
#Store Sales Rank

store_rank = (
    store_stats[["Store", "Store_Avg_Sales"]]
    .sort_values("Store_Avg_Sales", ascending=False)
)

store_rank["Store_Sales_Rank"] = range(1, len(store_rank) + 1)

store_rank.head()

,Store,Store_Avg_Sales,Store_Sales_Rank
19,20,29508.301592,1
3,4,29161.210415,2
13,14,28784.851727,3
12,13,27355.136891,4
1,2,26898.070031,5


In [125]:
#Merge Store Rank

train = train.merge(
    store_rank[["Store", "Store_Sales_Rank"]],
    on="Store",
    how="left"
)

In [126]:
#merge test data:

test = test.merge(
    features,
    on=["Store", "Date", "IsHoliday"],
    how="left"
)

In [127]:
test = test.merge(
    stores,
    on="Store",
    how="left"
)

In [128]:
#Encode Store Type

train["Store_Type"] = (
    train["Type"]
         .map({
             "A": 0,
             "B": 1,
             "C": 2
         })
)

In [129]:
#Relative Store Size

train["Store_Size_Relative"] = (
    train["Size"] / train["Size"].max()
)

In [130]:
#Verify Store Features

train[
    [
        "Store",
        "Type",
        "Store_Type",
        "Size",
        "Store_Size_Relative",
        "Store_Avg_Sales",
        "Store_Std_Sales",
        "Store_Min_Sales",
        "Store_Max_Sales",
        "Store_Sales_Rank"
    ]
].head()

,Store,Type,Store_Type,Size,Store_Size_Relative,Store_Avg_Sales,Store_Std_Sales,Store_Min_Sales,Store_Max_Sales,Store_Sales_Rank
0,1,A,0,151315,0.688979,21710.543621,27748.945511,-863.0,203670.47,9
1,1,A,0,151315,0.688979,21710.543621,27748.945511,-863.0,203670.47,9
2,1,A,0,151315,0.688979,21710.543621,27748.945511,-863.0,203670.47,9
3,1,A,0,151315,0.688979,21710.543621,27748.945511,-863.0,203670.47,9
4,1,A,0,151315,0.688979,21710.543621,27748.945511,-863.0,203670.47,9


In [131]:
#Missing Values

store_feature_columns = [
    "Store_Avg_Sales",
    "Store_Std_Sales",
    "Store_Min_Sales",
    "Store_Max_Sales",
    "Store_Sales_Rank",
    "Store_Type",
    "Store_Size_Relative"
]

train[store_feature_columns].isnull().sum()

Store_Avg_Sales        0
Store_Std_Sales        0
Store_Min_Sales        0
Store_Max_Sales        0
Store_Sales_Rank       0
Store_Type             0
Store_Size_Relative    0
dtype: int64

#### Department Features

In [132]:
#Average Sales by Department

department_stats = (
    train.groupby("Dept")
         .agg(
             Dept_Avg_Sales=("Weekly_Sales", "mean"),
             Dept_Std_Sales=("Weekly_Sales", "std"),
             Dept_Min_Sales=("Weekly_Sales", "min"),
             Dept_Max_Sales=("Weekly_Sales", "max")
         )
         .reset_index()
)

department_stats.head()

,Dept,Dept_Avg_Sales,Dept_Std_Sales,Dept_Min_Sales,Dept_Max_Sales
0,1,19213.485088,15102.373853,711.11,172225.55
1,2,43607.020113,25176.756920,5453.18,151090.50
2,3,11793.698516,12790.994371,2.00,131564.25
3,4,25974.630238,13261.140706,4695.19,72179.92
4,5,21365.583515,19988.452259,-0.04,259955.82


In [133]:
#Merge Department Statistics

train = train.merge(
    department_stats,
    on="Dept",
    how="left"
)

train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Year,Quarter,Month,Week,Day,DayOfWeek,IsWeekend,Lag_1,Lag_2,Lag_4,Lag_8,Lag_12,Lag_52,Rolling_Mean_4,Rolling_Mean_8,Rolling_Mean_12,Rolling_Std_4,Rolling_Std_8,Rolling_Std_12,Rolling_Min_4,Rolling_Max_4,Expanding_Mean,Expanding_Std,Expanding_Min,Expanding_Max,Holiday_Flag,Previous_Holiday,Next_Holiday,Near_Holiday,Store_Avg_Sales,Store_Std_Sales,Store_Min_Sales,Store_Max_Sales,Store_Sales_Rank,Store_Type,Store_Size_Relative,Dept_Avg_Sales,Dept_Std_Sales,Dept_Min_Sales,Dept_Max_Sales
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,2010,1,2,5,5,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,1,21710.543621,27748.945511,-863.0,203670.47,9,0,0.688979,19213.485088,15102.373853,711.11,172225.55
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,2010,1,2,6,12,4,0,24924.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24924.500000,NaN,24924.50,24924.50,1,0,0,1,21710.543621,27748.945511,-863.0,203670.47,9,0,0.688979,19213.485088,15102.373853,711.11,172225.55
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,2010,1,2,7,19,4,0,46039.49,24924.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35481.995000,14930.552614,24924.50,46039.49,0,1,0,1,21710.543621,27748.945511,-863.0,203670.47,9,0,0.688979,19213.485088,15102.373853,711.11,172225.55
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,2010,1,2,8,26,4,0,41595.55,46039.49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37519.846667,11131.900957,24924.50,46039.49,0,0,0,0,21710.543621,27748.945511,-863.0,203670.47,9,0,0.688979,19213.485088,15102.373853,711.11,172225.55
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,2010,1,3,9,5,4,0,19403.54,41595.55,24924.5,NaN,NaN,NaN,32990.77,NaN,NaN,12832.106391,NaN,NaN,19403.54,46039.49,32990.770000,12832.106391,19403.54,46039.49,0,0,0,0,21710.543621,27748.945511,-863.0,203670.47,9,0,0.688979,19213.485088,15102.373853,711.11,172225.55


In [134]:
#Department Sales Rank

department_rank = (
    department_stats[["Dept", "Dept_Avg_Sales"]]
    .sort_values("Dept_Avg_Sales", ascending=False)
)

department_rank["Dept_Sales_Rank"] = range(
    1,
    len(department_rank) + 1
)

department_rank.head()

,Dept,Dept_Avg_Sales,Dept_Sales_Rank
73,92,75204.870531,1
76,95,69824.423080,2
36,38,61090.619568,3
60,72,50566.515417,4
57,65,45441.706224,5


In [135]:
#Merge Department Rank

train = train.merge(
    department_rank[
        ["Dept", "Dept_Sales_Rank"]
    ],
    on="Dept",
    how="left"
)

In [136]:
#Department Sales Contribution

department_contribution = (
    train.groupby("Dept")["Weekly_Sales"]
         .sum()
         .reset_index(name="Dept_Total_Sales")
)

department_contribution["Dept_Sales_Contribution"] = (
    department_contribution["Dept_Total_Sales"]
    / department_contribution["Dept_Total_Sales"].sum()
) * 100

department_contribution.head()

,Dept,Dept_Total_Sales,Dept_Sales_Contribution
0,1,1.236388e+08,1.835160
1,2,2.806112e+08,4.165089
2,3,7.589245e+07,1.126466
3,4,1.671467e+08,2.480946
4,5,1.356074e+08,2.012809


In [137]:
#Merge Department Contribution

train = train.merge(
    department_contribution[
        ["Dept", "Dept_Sales_Contribution"]
    ],
    on="Dept",
    how="left"
)

In [138]:
#Verify Department Features

train[
    [
        "Dept",
        "Dept_Avg_Sales",
        "Dept_Std_Sales",
        "Dept_Min_Sales",
        "Dept_Max_Sales",
        "Dept_Sales_Rank",
        "Dept_Sales_Contribution"
    ]
].head()

,Dept,Dept_Avg_Sales,Dept_Std_Sales,Dept_Min_Sales,Dept_Max_Sales,Dept_Sales_Rank,Dept_Sales_Contribution
0,1,19213.485088,15102.373853,711.11,172225.55,21,1.83516
1,1,19213.485088,15102.373853,711.11,172225.55,21,1.83516
2,1,19213.485088,15102.373853,711.11,172225.55,21,1.83516
3,1,19213.485088,15102.373853,711.11,172225.55,21,1.83516
4,1,19213.485088,15102.373853,711.11,172225.55,21,1.83516


In [139]:
#Check Missing Values

department_feature_columns = [
    "Dept_Avg_Sales",
    "Dept_Std_Sales",
    "Dept_Min_Sales",
    "Dept_Max_Sales",
    "Dept_Sales_Rank",
    "Dept_Sales_Contribution"
]

train[department_feature_columns].isnull().sum()

Dept_Avg_Sales             0
Dept_Std_Sales             0
Dept_Min_Sales             0
Dept_Max_Sales             0
Dept_Sales_Rank            0
Dept_Sales_Contribution    0
dtype: int64

#### Cyclical Encoding

In [140]:
#Encode Month

train["Month_Sin"] = np.sin(
    2 * np.pi * train["Month"] / 12
)

train["Month_Cos"] = np.cos(
    2 * np.pi * train["Month"] / 12
)

In [141]:
#Encode Week of Year

train["Week_Sin"] = np.sin(
    2 * np.pi * train["Week"] / 52
)

train["Week_Cos"] = np.cos(
    2 * np.pi * train["Week"] / 52
)

In [142]:
#Encode Quarter

train["Quarter_Sin"] = np.sin(
    2 * np.pi * train["Quarter"] / 4
)

train["Quarter_Cos"] = np.cos(
    2 * np.pi * train["Quarter"] / 4
)

In [143]:
#Encode Day of Week

train["DayOfWeek_Sin"] = np.sin(
    2 * np.pi * train["DayOfWeek"] / 7
)

train["DayOfWeek_Cos"] = np.cos(
    2 * np.pi * train["DayOfWeek"] / 7
)

In [144]:
#Verify Encoded Features

train[
    [
        "Month",
        "Month_Sin",
        "Month_Cos",
        "Week",
        "Week_Sin",
        "Week_Cos",
        "Quarter",
        "Quarter_Sin",
        "Quarter_Cos"
    ]
].head()

,Month,Month_Sin,Month_Cos,Week,Week_Sin,Week_Cos,Quarter,Quarter_Sin,Quarter_Cos
0,2,0.866025,5.000000e-01,5,0.568065,0.822984,1,1.0,6.123234e-17
1,2,0.866025,5.000000e-01,6,0.663123,0.748511,1,1.0,6.123234e-17
2,2,0.866025,5.000000e-01,7,0.748511,0.663123,1,1.0,6.123234e-17
3,2,0.866025,5.000000e-01,8,0.822984,0.568065,1,1.0,6.123234e-17
4,3,1.000000,6.123234e-17,9,0.885456,0.464723,1,1.0,6.123234e-17


In [145]:
#Check Missing Values

cyclical_columns = [
    "Month_Sin",
    "Month_Cos",
    "Week_Sin",
    "Week_Cos",
    "Quarter_Sin",
    "Quarter_Cos",
    "DayOfWeek_Sin",
    "DayOfWeek_Cos"
]

train[cyclical_columns].isnull().sum()

Month_Sin        0
Month_Cos        0
Week_Sin         0
Week_Cos         0
Quarter_Sin      0
Quarter_Cos      0
DayOfWeek_Sin    0
DayOfWeek_Cos    0
dtype: int64

#### Handling Missing Values

In [146]:
#Check Missing Values

missing_values = (
    train.isnull()
         .sum()
         .sort_values(ascending=False)
)

missing_values[missing_values > 0]

Lag_52             160487
Lag_12              38615
Rolling_Mean_12     38615
Rolling_Std_12      38615
Rolling_Std_8       25966
Rolling_Mean_8      25966
Lag_8               25966
Rolling_Max_4       13134
Rolling_Std_4       13134
Rolling_Mean_4      13134
Rolling_Min_4       13134
Lag_4               13134
Expanding_Std        6625
Lag_2                6625
Expanding_Min        3331
Expanding_Max        3331
Lag_1                3331
Expanding_Mean       3331
dtype: int64

In [147]:
#Missing Value Percentage

missing_percentage = (
    train.isnull()
         .mean()
         .mul(100)
         .round(2)
         .sort_values(ascending=False)
)

missing_percentage[missing_percentage > 0]

Lag_52             38.07
Lag_12              9.16
Rolling_Mean_12     9.16
Rolling_Std_12      9.16
Rolling_Std_8       6.16
Rolling_Mean_8      6.16
Lag_8               6.16
Rolling_Max_4       3.12
Rolling_Std_4       3.12
Rolling_Mean_4      3.12
Rolling_Min_4       3.12
Lag_4               3.12
Expanding_Std       1.57
Lag_2               1.57
Expanding_Min       0.79
Expanding_Max       0.79
Lag_1               0.79
Expanding_Mean      0.79
dtype: float64

In [148]:
#Identify Engineered Features with Missing Values

engineered_columns = [
    column
    for column in train.columns
    if (
        "Lag_" in column
        or "Rolling_" in column
        or "Expanding_" in column
    )
]

train[engineered_columns].isnull().sum()

Lag_1                3331
Lag_2                6625
Lag_4               13134
Lag_8               25966
Lag_12              38615
Lag_52             160487
Rolling_Mean_4      13134
Rolling_Mean_8      25966
Rolling_Mean_12     38615
Rolling_Std_4       13134
Rolling_Std_8       25966
Rolling_Std_12      38615
Rolling_Min_4       13134
Rolling_Max_4       13134
Expanding_Mean       3331
Expanding_Std        6625
Expanding_Min        3331
Expanding_Max        3331
dtype: int64

In [149]:
#Fill Missing Values

train[engineered_columns] = (
    train[engineered_columns]
    .fillna(0)
)

In [150]:
#Verify Missing Values

train.isnull().sum().sum()

np.int64(0)

In [151]:
#Verify Engineered Features

train[
    engineered_columns
].head()

,Lag_1,Lag_2,Lag_4,Lag_8,Lag_12,Lag_52,Rolling_Mean_4,Rolling_Mean_8,Rolling_Mean_12,Rolling_Std_4,Rolling_Std_8,Rolling_Std_12,Rolling_Min_4,Rolling_Max_4,Expanding_Mean,Expanding_Std,Expanding_Min,Expanding_Max
0,0.00,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.000000,0.0,0.0,0.00,0.00,0.000000,0.000000,0.00,0.00
1,24924.50,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.000000,0.0,0.0,0.00,0.00,24924.500000,0.000000,24924.50,24924.50
2,46039.49,24924.50,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.000000,0.0,0.0,0.00,0.00,35481.995000,14930.552614,24924.50,46039.49
3,41595.55,46039.49,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.000000,0.0,0.0,0.00,0.00,37519.846667,11131.900957,24924.50,46039.49
4,19403.54,41595.55,24924.5,0.0,0.0,0.0,32990.77,0.0,0.0,12832.106391,0.0,0.0,19403.54,46039.49,32990.770000,12832.106391,19403.54,46039.49


In [152]:
#Dataset Shape

train.shape

(421570, 66)

In [153]:
#Data Types

train.info()

<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 66 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   Store                    421570 non-null  int64         
 1   Dept                     421570 non-null  int64         
 2   Date                     421570 non-null  datetime64[us]
 3   Weekly_Sales             421570 non-null  float64       
 4   IsHoliday                421570 non-null  bool          
 5   Type                     421570 non-null  str           
 6   Size                     421570 non-null  int64         
 7   Temperature              421570 non-null  float64       
 8   Fuel_Price               421570 non-null  float64       
 9   MarkDown1                421570 non-null  float64       
 10  MarkDown2                421570 non-null  float64       
 11  MarkDown3                421570 non-null  float64       
 12  MarkDown4                42

#### Feature Selection

In [154]:
#Target Variable

target = "Weekly_Sales"

target

'Weekly_Sales'

In [155]:
#Select Feature Columns

feature_columns = [

    # Store Information
    "Store",
    "Dept",
    "Type",
    "Size",

    # Date Features
    "Year",
    "Quarter",
    "Month",
    "Week",
    "Day",
    "DayOfWeek",
    "IsWeekend",

    # Holiday Features
    "Holiday_Flag",
    "Previous_Holiday",
    "Next_Holiday",
    "Near_Holiday",

    # External Features
    "Temperature",
    "Fuel_Price",
    "CPI",
    "Unemployment",

    # Lag Features
    "Lag_1",
    "Lag_2",
    "Lag_4",
    "Lag_8",
    "Lag_12",
    "Lag_52",

    # Rolling Features
    "Rolling_Mean_4",
    "Rolling_Mean_8",
    "Rolling_Mean_12",
    "Rolling_Std_4",
    "Rolling_Std_8",
    "Rolling_Std_12",
    "Rolling_Min_4",
    "Rolling_Max_4",

    # Expanding Features
    "Expanding_Mean",
    "Expanding_Std",
    "Expanding_Min",
    "Expanding_Max",

    # Store Features
    "Store_Avg_Sales",
    "Store_Std_Sales",
    "Store_Min_Sales",
    "Store_Max_Sales",
    "Store_Sales_Rank",
    "Store_Type",
    "Store_Size_Relative",

    # Department Features
    "Dept_Avg_Sales",
    "Dept_Std_Sales",
    "Dept_Min_Sales",
    "Dept_Max_Sales",
    "Dept_Sales_Rank",
    "Dept_Sales_Contribution",

    # Cyclical Features
    "Month_Sin",
    "Month_Cos",
    "Week_Sin",
    "Week_Cos",
    "Quarter_Sin",
    "Quarter_Cos",
    "DayOfWeek_Sin",
    "DayOfWeek_Cos"
]

In [156]:
#Create Feature Matrix and Target

X = train[feature_columns]

y = train[target]

In [157]:
#Verify Dataset

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (421570, 58)
Target Shape: (421570,)


In [158]:
#Preview Features

X.head()

,Store,Dept,Type,Size,Year,Quarter,Month,Week,Day,DayOfWeek,IsWeekend,Holiday_Flag,Previous_Holiday,Next_Holiday,Near_Holiday,Temperature,Fuel_Price,CPI,Unemployment,Lag_1,Lag_2,Lag_4,Lag_8,Lag_12,Lag_52,Rolling_Mean_4,Rolling_Mean_8,Rolling_Mean_12,Rolling_Std_4,Rolling_Std_8,Rolling_Std_12,Rolling_Min_4,Rolling_Max_4,Expanding_Mean,Expanding_Std,Expanding_Min,Expanding_Max,Store_Avg_Sales,Store_Std_Sales,Store_Min_Sales,Store_Max_Sales,Store_Sales_Rank,Store_Type,Store_Size_Relative,Dept_Avg_Sales,Dept_Std_Sales,Dept_Min_Sales,Dept_Max_Sales,Dept_Sales_Rank,Dept_Sales_Contribution,Month_Sin,Month_Cos,Week_Sin,Week_Cos,Quarter_Sin,Quarter_Cos,DayOfWeek_Sin,DayOfWeek_Cos
0,1,1,A,151315,2010,1,2,5,5,4,0,0,0,1,1,42.31,2.572,211.096358,8.106,0.00,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.000000,0.0,0.0,0.00,0.00,0.000000,0.000000,0.00,0.00,21710.543621,27748.945511,-863.0,203670.47,9,0,0.688979,19213.485088,15102.373853,711.11,172225.55,21,1.83516,0.866025,5.000000e-01,0.568065,0.822984,1.0,6.123234e-17,-0.433884,-0.900969
1,1,1,A,151315,2010,1,2,6,12,4,0,1,0,0,1,38.51,2.548,211.242170,8.106,24924.50,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.000000,0.0,0.0,0.00,0.00,24924.500000,0.000000,24924.50,24924.50,21710.543621,27748.945511,-863.0,203670.47,9,0,0.688979,19213.485088,15102.373853,711.11,172225.55,21,1.83516,0.866025,5.000000e-01,0.663123,0.748511,1.0,6.123234e-17,-0.433884,-0.900969
2,1,1,A,151315,2010,1,2,7,19,4,0,0,1,0,1,39.93,2.514,211.289143,8.106,46039.49,24924.50,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.000000,0.0,0.0,0.00,0.00,35481.995000,14930.552614,24924.50,46039.49,21710.543621,27748.945511,-863.0,203670.47,9,0,0.688979,19213.485088,15102.373853,711.11,172225.55,21,1.83516,0.866025,5.000000e-01,0.748511,0.663123,1.0,6.123234e-17,-0.433884,-0.900969
3,1,1,A,151315,2010,1,2,8,26,4,0,0,0,0,0,46.63,2.561,211.319643,8.106,41595.55,46039.49,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.000000,0.0,0.0,0.00,0.00,37519.846667,11131.900957,24924.50,46039.49,21710.543621,27748.945511,-863.0,203670.47,9,0,0.688979,19213.485088,15102.373853,711.11,172225.55,21,1.83516,0.866025,5.000000e-01,0.822984,0.568065,1.0,6.123234e-17,-0.433884,-0.900969
4,1,1,A,151315,2010,1,3,9,5,4,0,0,0,0,0,46.50,2.625,211.350143,8.106,19403.54,41595.55,24924.5,0.0,0.0,0.0,32990.77,0.0,0.0,12832.106391,0.0,0.0,19403.54,46039.49,32990.770000,12832.106391,19403.54,46039.49,21710.543621,27748.945511,-863.0,203670.47,9,0,0.688979,19213.485088,15102.373853,711.11,172225.55,21,1.83516,1.000000,6.123234e-17,0.885456,0.464723,1.0,6.123234e-17,-0.433884,-0.900969


In [159]:
#Feature List

pd.DataFrame({
    "Feature": feature_columns
})

,Feature
0,Store
1,Dept
2,Type
3,Size
4,Year
5,Quarter
6,Month
7,Week
8,Day
9,DayOfWeek


In [160]:
#Save Full Dataset

train.to_csv(
    "../data/processed/train_feature_engineered.csv",
    index=False
)

In [161]:
#Save Feature Matrix

X.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

In [162]:
#Save Target

y.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

In [163]:
#Verify Saved Files

import os

files = [
    "../data/processed/train_feature_engineered.csv",
    "../data/processed/X_train.csv",
    "../data/processed/y_train.csv"
]

for file in files:
    print(file, "Exists:", os.path.exists(file))

../data/processed/train_feature_engineered.csv Exists: True
../data/processed/X_train.csv Exists: True
../data/processed/y_train.csv Exists: True


In [165]:
test["Date"] = pd.to_datetime(test["Date_x"])

In [168]:
# CALENDAR FEATURES

test["Year"] = test["Date"].dt.year

test["Quarter"] = test["Date"].dt.quarter

test["Month"] = test["Date"].dt.month

test["Week"] = (
    test["Date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

test["Day"] = test["Date"].dt.day

test["DayOfWeek"] = test["Date"].dt.dayofweek

test["IsWeekend"] = (
    test["DayOfWeek"] >= 5
).astype(int)

In [167]:
# HOLIDAY FEATURES

test["Holiday_Flag"] = (
    test["IsHoliday"].astype(int)
)

In [169]:
test = test.sort_values(
    ["Store", "Dept", "Date"]
).reset_index(drop=True)

In [170]:
test["Previous_Holiday"] = (
    test
    .groupby(["Store", "Dept"])["Holiday_Flag"]
    .shift(1)
    .fillna(0)
)

test["Next_Holiday"] = (
    test
    .groupby(["Store", "Dept"])["Holiday_Flag"]
    .shift(-1)
    .fillna(0)
)

test["Near_Holiday"] = (
    (
        test["Previous_Holiday"] == 1
    )
    |
    (
        test["Next_Holiday"] == 1
    )
).astype(int)

In [171]:
# CYCLICAL FEATURES

test["Month_Sin"] = np.sin(
    2 * np.pi * test["Month"] / 12
)

test["Month_Cos"] = np.cos(
    2 * np.pi * test["Month"] / 12
)

test["Week_Sin"] = np.sin(
    2 * np.pi * test["Week"] / 52
)

test["Week_Cos"] = np.cos(
    2 * np.pi * test["Week"] / 52
)

test["Quarter_Sin"] = np.sin(
    2 * np.pi * test["Quarter"] / 4
)

test["Quarter_Cos"] = np.cos(
    2 * np.pi * test["Quarter"] / 4
)

test["DayOfWeek_Sin"] = np.sin(
    2 * np.pi * test["DayOfWeek"] / 7
)

test["DayOfWeek_Cos"] = np.cos(
    2 * np.pi * test["DayOfWeek"] / 7
)

In [172]:
# HISTORICAL SALES

sales_history = train[
    [
        "Store",
        "Dept",
        "Date",
        "Weekly_Sales"
    ]
].copy()

sales_history = sales_history.sort_values(
    ["Store", "Dept", "Date"]
)

print(sales_history.shape)

(421570, 4)


In [173]:
# LAG FEATURES


for lag in [1, 2, 4, 8, 12, 52]:

    lag_data = sales_history[
        [
            "Store",
            "Dept",
            "Date",
            "Weekly_Sales"
        ]
    ].copy()

    lag_data["Date"] = (
        lag_data["Date"]
        + pd.Timedelta(weeks=lag)
    )

    lag_data = lag_data.rename(
        columns={
            "Weekly_Sales": f"Lag_{lag}"
        }
    )

    test = test.merge(
        lag_data,
        on=["Store", "Dept", "Date"],
        how="left"
    )

In [174]:
test[
    [
        "Store",
        "Dept",
        "Date",
        "Lag_1",
        "Lag_2",
        "Lag_4",
        "Lag_8",
        "Lag_12",
        "Lag_52"
    ]
].head()

,Store,Dept,Date,Lag_1,Lag_2,Lag_4,Lag_8,Lag_12,Lag_52
0,1,1,2012-11-02,27390.81,24185.27,21904.47,18322.37,16119.92,39886.06
1,1,1,2012-11-02,27390.81,24185.27,21904.47,18322.37,16119.92,39886.06
2,1,1,2012-11-02,27390.81,24185.27,21904.47,18322.37,16119.92,39886.06
3,1,1,2012-11-02,27390.81,24185.27,21904.47,18322.37,16119.92,39886.06
4,1,1,2012-11-02,27390.81,24185.27,21904.47,18322.37,16119.92,39886.06


In [175]:
# ROLLING / EXPANDING FEATURES

def get_history(store, dept, date):

    return sales_history[
        (sales_history["Store"] == store)
        &
        (sales_history["Dept"] == dept)
        &
        (sales_history["Date"] < date)
    ]["Weekly_Sales"]

In [177]:
for (store, dept), group in sales_history.groupby(["Store", "Dept"]):

    sales = group.sort_values("Date")["Weekly_Sales"]

    mask = (
        (test["Store"] == store) &
        (test["Dept"] == dept)
    )

    test.loc[mask, "Rolling_Mean_4"] = sales.tail(4).mean()
    test.loc[mask, "Rolling_Mean_8"] = sales.tail(8).mean()
    test.loc[mask, "Rolling_Mean_12"] = sales.tail(12).mean()

    test.loc[mask, "Rolling_Std_4"] = sales.tail(4).std()
    test.loc[mask, "Rolling_Std_8"] = sales.tail(8).std()
    test.loc[mask, "Rolling_Std_12"] = sales.tail(12).std()

    test.loc[mask, "Rolling_Min_4"] = sales.tail(4).min()
    test.loc[mask, "Rolling_Max_4"] = sales.tail(4).max()

print("Rolling features created successfully.")

Rolling features created successfully.


In [178]:
# EXPANDING FEATURES

expanding_features = (
    sales_history
    .groupby(["Store", "Dept"])["Weekly_Sales"]
    .agg(
        Expanding_Mean="mean",
        Expanding_Std="std",
        Expanding_Min="min",
        Expanding_Max="max"
    )
    .reset_index()
)

# Merge expanding features into test
test = test.merge(
    expanding_features,
    on=["Store", "Dept"],
    how="left"
)

print("Expanding features created successfully.")

Expanding features created successfully.


In [179]:
# STORE FEATURES

store_stats = (
    train
    .groupby("Store")["Weekly_Sales"]
    .agg(
        Store_Avg_Sales="mean",
        Store_Std_Sales="std",
        Store_Min_Sales="min",
        Store_Max_Sales="max"
    )
    .reset_index()
)

In [180]:
#Store rank:

store_stats["Store_Sales_Rank"] = (
    store_stats["Store_Avg_Sales"]
    .rank(
        ascending=False,
        method="min"
    )
)

In [181]:
#Merge:

test = test.merge(
    store_stats,
    on="Store",
    how="left"
)

In [182]:
#Store type:

test["Store_Type"] = (
    test["Type"]
    .map({
        "A": 1,
        "B": 2,
        "C": 3
    })
)

In [183]:
#Store relative size:

test["Store_Size_Relative"] = (
    test["Size"]
    / stores["Size"].mean()
)

In [184]:
# DEPARTMENT FEATURES

dept_stats = (
    train
    .groupby("Dept")["Weekly_Sales"]
    .agg(
        Dept_Avg_Sales="mean",
        Dept_Std_Sales="std",
        Dept_Min_Sales="min",
        Dept_Max_Sales="max"
    )
    .reset_index()
)

In [185]:
#Department rank:

dept_stats["Dept_Sales_Rank"] = (
    dept_stats["Dept_Avg_Sales"]
    .rank(
        ascending=False,
        method="min"
    )
)

In [186]:
#Department contribution:

dept_stats["Dept_Sales_Contribution"] = (
    dept_stats["Dept_Avg_Sales"]
    / dept_stats["Dept_Avg_Sales"].sum()
)

In [187]:
#Merge:

test = test.merge(
    dept_stats,
    on="Dept",
    how="left"
)

In [188]:
feature_columns = [
    'Store', 'Dept', 'Year', 'Quarter', 'Month', 'Week',
    'Day', 'DayOfWeek', 'IsWeekend', 'Holiday_Flag',
    'Previous_Holiday', 'Next_Holiday', 'Near_Holiday',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'Lag_1', 'Lag_2', 'Lag_4', 'Lag_8', 'Lag_12', 'Lag_52',
    'Rolling_Mean_4', 'Rolling_Mean_8', 'Rolling_Mean_12',
    'Rolling_Std_4', 'Rolling_Std_8', 'Rolling_Std_12',
    'Rolling_Min_4', 'Rolling_Max_4',
    'Expanding_Mean', 'Expanding_Std',
    'Expanding_Min', 'Expanding_Max',
    'Store_Avg_Sales', 'Store_Std_Sales',
    'Store_Min_Sales', 'Store_Max_Sales',
    'Store_Sales_Rank', 'Store_Type',
    'Store_Size_Relative',
    'Dept_Avg_Sales', 'Dept_Std_Sales',
    'Dept_Min_Sales', 'Dept_Max_Sales',
    'Dept_Sales_Rank', 'Dept_Sales_Contribution',
    'Month_Sin', 'Month_Cos',
    'Week_Sin', 'Week_Cos',
    'Quarter_Sin', 'Quarter_Cos',
    'DayOfWeek_Sin', 'DayOfWeek_Cos'
]

In [189]:
missing_features = [
    col
    for col in feature_columns
    if col not in test.columns
]

print("Missing Features:")
print(missing_features)

Missing Features:
[]


In [190]:
test_feature_engineered = test[
    [
        "Store",
        "Dept",
        "Date",
        "IsHoliday"
    ] + feature_columns
].copy()

In [191]:
print(
    "Test Feature Shape:",
    test_feature_engineered.shape
)

Test Feature Shape: (468577, 60)


In [193]:
#Save test_feature_engineered.csv

test_feature_engineered.to_csv(
    "../data/processed/test_feature_engineered.csv",
    index=False
)

In [194]:
import os

print(
    os.path.exists(
        "../data/processed/test_feature_engineered.csv"
    )
)

True


In [195]:
test_feature_engineered.head()

,Store,Dept,Date,IsHoliday,Store,Dept,Year,Quarter,Month,Week,Day,DayOfWeek,IsWeekend,Holiday_Flag,Previous_Holiday,Next_Holiday,Near_Holiday,Temperature,Fuel_Price,CPI,Unemployment,Lag_1,Lag_2,Lag_4,Lag_8,Lag_12,Lag_52,Rolling_Mean_4,Rolling_Mean_8,Rolling_Mean_12,Rolling_Std_4,Rolling_Std_8,Rolling_Std_12,Rolling_Min_4,Rolling_Max_4,Expanding_Mean,Expanding_Std,Expanding_Min,Expanding_Max,Store_Avg_Sales,Store_Std_Sales,Store_Min_Sales,Store_Max_Sales,Store_Sales_Rank,Store_Type,Store_Size_Relative,Dept_Avg_Sales,Dept_Std_Sales,Dept_Min_Sales,Dept_Max_Sales,Dept_Sales_Rank,Dept_Sales_Contribution,Month_Sin,Month_Cos,Week_Sin,Week_Cos,Quarter_Sin,Quarter_Cos,DayOfWeek_Sin,DayOfWeek_Cos
0,1,1,2012-11-02,False,1,1,2012,4,11,44,2,4,0,0,0.0,0.0,0,42.31,2.572,211.096358,8.106,27390.81,24185.27,21904.47,18322.37,16119.92,39886.06,24061.14,21547.8075,19899.976667,2410.800855,3136.666469,3502.05341,21904.47,27390.81,22513.322937,9854.349032,14537.37,57592.12,21710.543621,27748.945511,-863.0,203670.47,9.0,1,1.161392,19213.485088,15102.373853,711.11,172225.55,21.0,0.016905,-0.5,0.866025,-0.822984,0.568065,-2.449294e-16,1.0,-0.433884,-0.900969
1,1,1,2012-11-02,False,1,1,2012,4,11,44,2,4,0,0,0.0,0.0,0,58.74,2.689,211.956394,7.838,27390.81,24185.27,21904.47,18322.37,16119.92,39886.06,24061.14,21547.8075,19899.976667,2410.800855,3136.666469,3502.05341,21904.47,27390.81,22513.322937,9854.349032,14537.37,57592.12,21710.543621,27748.945511,-863.0,203670.47,9.0,1,1.161392,19213.485088,15102.373853,711.11,172225.55,21.0,0.016905,-0.5,0.866025,-0.822984,0.568065,-2.449294e-16,1.0,-0.433884,-0.900969
2,1,1,2012-11-02,False,1,1,2012,4,11,44,2,4,0,0,0.0,0.0,0,91.65,3.684,215.544618,7.962,27390.81,24185.27,21904.47,18322.37,16119.92,39886.06,24061.14,21547.8075,19899.976667,2410.800855,3136.666469,3502.05341,21904.47,27390.81,22513.322937,9854.349032,14537.37,57592.12,21710.543621,27748.945511,-863.0,203670.47,9.0,1,1.161392,19213.485088,15102.373853,711.11,172225.55,21.0,0.016905,-0.5,0.866025,-0.822984,0.568065,-2.449294e-16,1.0,-0.433884,-0.900969
3,1,1,2012-11-02,False,1,1,2012,4,11,44,2,4,0,0,0.0,0.0,0,75.55,3.749,221.671800,7.143,27390.81,24185.27,21904.47,18322.37,16119.92,39886.06,24061.14,21547.8075,19899.976667,2410.800855,3136.666469,3502.05341,21904.47,27390.81,22513.322937,9854.349032,14537.37,57592.12,21710.543621,27748.945511,-863.0,203670.47,9.0,1,1.161392,19213.485088,15102.373853,711.11,172225.55,21.0,0.016905,-0.5,0.866025,-0.822984,0.568065,-2.449294e-16,1.0,-0.433884,-0.900969
4,1,1,2012-11-02,False,1,1,2012,4,11,44,2,4,0,0,0.0,0.0,0,56.46,3.244,224.235290,6.525,27390.81,24185.27,21904.47,18322.37,16119.92,39886.06,24061.14,21547.8075,19899.976667,2410.800855,3136.666469,3502.05341,21904.47,27390.81,22513.322937,9854.349032,14537.37,57592.12,21710.543621,27748.945511,-863.0,203670.47,9.0,1,1.161392,19213.485088,15102.373853,711.11,172225.55,21.0,0.016905,-0.5,0.866025,-0.822984,0.568065,-2.449294e-16,1.0,-0.433884,-0.900969


### Key Observations

- A comprehensive set of predictive features has been engineered from historical sales, calendar information, store characteristics, department characteristics, holidays, and external variables.
- Lag, rolling, and expanding features capture both short-term and long-term demand patterns.
- Calendar-based and cyclical features represent seasonal behavior effectively.
- The dataset contains no missing values and is ready for model development.
- Feature engineering has transformed the raw transactional data into a structured modeling dataset suitable for machine learning and time series forecasting.

### Executive Summary

The feature engineering process successfully transformed Walmart's historical sales data into a high-quality modeling dataset. New features were created to represent temporal patterns, historical sales behavior, holiday effects, store characteristics, department performance, and seasonal cycles.

These engineered features provide forecasting models with richer information than the original dataset, enabling them to capture complex demand patterns across stores and departments. The final dataset is now prepared for baseline forecasting models as well as advanced machine learning algorithms such as Prophet, XGBoost, and LightGBM.